# Seasonal Decomposition and Model Development

In [58]:
# needed for data handling
import pandas as pd

# needed for time series seasonal decomposition analysis
from statsmodels.tsa.seasonal import seasonal_decompose

# needed to visualize the seasonal decomposition analysis
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# needed for constructing the ARIMA models
from statsmodels.tsa.arima.model import ARIMA

##### For the purpose of studying the seasonal decomposition of temperature and snow depth, as well as the development of the ARIMA models, I chose one weather station to focus on. I chose a station in Newberry, Michigan. It is a town in the upper peninsula of Michigan that exhibits a large temperature range (it gets quite cold in the winter and nicely warm in the summer) and they consistently get a relatively large amount of snow in the winter. Those factors should make it easier to see the seasonality in both temperature and snow depth. I used the data from this weather station to tune the parameters used in the ARIMA models for all the weather stations in this project.

In [59]:
# USC00202598 is the weather station in Newberry, Michigan that I am using for this analysis.
# This test file is derived from the CSV file that I downloaded from the NOAA website for the weather station USC00202598
# with all extraneous columns removed, only keeping the date, max temperature, and snow depth data columns.
# Also, the daily values have been aggregated to obtain monthly averages for each of those columns.
file = 'USC00202598test.csv'
ws_df = pd.read_csv(file, low_memory=False, parse_dates=['DATE'])

# dataframe for temperature
tmax_df = ws_df[['DATE','TMAX']].copy().dropna()

# dataframe for snow depth
snwd_df = ws_df[['DATE','SNWD']].copy().dropna()

# Attribution Note 

I borrowed, heavily, from https://www.kaggle.com/code/mateuszk013/forecasting-weather-patterns-with-arima-rnn for the plotting of the seasonal decomposition components.

### Seasonal Decomposition - Temperature

In [60]:
decomposition = seasonal_decompose(tmax_df.TMAX, model='additive', period=12)
PLOT_COLOR = '#161e54'
BACKGROUND_COLOR = 'ghostwhite'

fig = make_subplots(
    rows=4,
    cols=1,
    vertical_spacing=0.1,
    x_title='Month',
    y_title='Temperature (\u2109)',
    subplot_titles=['Observed Values', 'Trend', 'Seasonality', 'Residuals']
)

observed = go.Scatter(
    x=decomposition.observed.index,
    y=decomposition.observed,
    line=dict(color=PLOT_COLOR),
)
trend = go.Scatter(
    x=decomposition.trend.index,
    y=decomposition.trend,
    line=dict(color=PLOT_COLOR),
)
seasonal = go.Scatter(
    x=decomposition.seasonal.index,
    y=decomposition.seasonal,
    line=dict(color=PLOT_COLOR),
)
residuals = go.Scatter(
    x=decomposition.resid.index,
    y=decomposition.resid,
    mode='markers',
    marker_size=2,
    line=dict(color=PLOT_COLOR),
)

fig.add_trace(observed, row=1, col=1)
fig.add_trace(trend, row=2, col=1)
fig.add_trace(seasonal, row=3, col=1)
fig.add_trace(residuals, row=4, col=1)

fig.update_annotations(font_size=14)
fig.update_layout(
    font_color=PLOT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    title_text='Seasonal Decomposition - Temperature',
    showlegend=False,
    height=800,
    width=840
)
fig.update_xaxes(dtick=12)
fig.show()

### Model Development - Temperature

In [61]:
train_df = tmax_df.copy()
train_df = train_df.set_index('DATE')
train_df.index = pd.DatetimeIndex(train_df.index).to_period('M')
tmax_model = ARIMA(train_df, order=(1,0,1), seasonal_order=(0,1,1,12)).fit()
tmax_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                    SARIMAX Results                                     
========================================================================================
Dep. Variable:                             TMAX   No. Observations:                  180
Model:             ARIMA(1, 0, 1)x(0, 1, 1, 12)   Log Likelihood                -464.749
Date:                          Sat, 22 Mar 2025   AIC                            937.498
Time:                                  11:06:56   BIC                            949.994
Sample:                              01-31-2010   HQIC                           942.570
                                   - 12-31-2024                                         
Covariance Type:                            opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.7751      0.129      5.997      0.000       0.522       1.028
ma.L1         -0.4429      0.179     -2.471      0.013      -0.794      -0.092
ma.S.L12      -0.9997     37.574     -0.027      0.979     -74.644      72.645
sigma2        12.1891    457.539      0.027      0.979    -884.571     908.949
===================================================================================
Ljung-Box (L1) (Q):                   0.02   Jarque-Bera (JB):                 4.04
Prob(Q):                              0.90   Prob(JB):                         0.13
Heteroskedasticity (H):               0.63   Skew:                            -0.34
Prob(H) (two-sided):                  0.09   Kurtosis:                         3.33
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

In [62]:
### Predictions for average max temp per month in 2025
arima_tmax_forecast = tmax_model.predict(start='2025-01-01', end='2025-12-31')
print(arima_tmax_forecast)

2025-01    26.916333
2025-02    27.893923
2025-03    37.979786
2025-04    48.606050
2025-05    63.746382
2025-06    72.877327
2025-07    78.646854
2025-08    76.911779
2025-09    69.286061
2025-10    55.296527
2025-11    42.442523
2025-12    32.193598
Freq: M, Name: predicted_mean, dtype: float64


### Seasonal Decomposition - Snow Depth

In [63]:
decomposition = seasonal_decompose(snwd_df.SNWD, model='additive', period=12)
PLOT_COLOR = '#161e54'
BACKGROUND_COLOR = 'ghostwhite'

fig = make_subplots(
    rows=4,
    cols=1,
    vertical_spacing=0.1,
    x_title='Month',
    y_title='Temperature (\u2109)',
    subplot_titles=['Observed Values', 'Trend', 'Seasonality', 'Residuals']
)

observed = go.Scatter(
    x=decomposition.observed.index,
    y=decomposition.observed,
    line=dict(color=PLOT_COLOR),
)
trend = go.Scatter(
    x=decomposition.trend.index,
    y=decomposition.trend,
    line=dict(color=PLOT_COLOR),
)
seasonal = go.Scatter(
    x=decomposition.seasonal.index,
    y=decomposition.seasonal,
    line=dict(color=PLOT_COLOR),
)
residuals = go.Scatter(
    x=decomposition.resid.index,
    y=decomposition.resid,
    mode='markers',
    marker_size=2,
    line=dict(color=PLOT_COLOR),
)

fig.add_trace(observed, row=1, col=1)
fig.add_trace(trend, row=2, col=1)
fig.add_trace(seasonal, row=3, col=1)
fig.add_trace(residuals, row=4, col=1)

fig.update_annotations(font_size=14)
fig.update_layout(
    font_color=PLOT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    title_text='Seasonal Decomposition - Temperature',
    showlegend=False,
    height=800,
    width=840
)
fig.update_xaxes(dtick=12)
fig.show()

### Model Development - Snow Depth

In [64]:
train_df = snwd_df.copy()
train_df = train_df.set_index('DATE')
train_df.index = pd.DatetimeIndex(train_df.index).to_period('M')
snwd_model = ARIMA(train_df, order=(2,0,0), seasonal_order=(2,1,1,12)).fit()
snwd_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                     SARIMAX Results                                      
==========================================================================================
Dep. Variable:                               SNWD   No. Observations:                  180
Model:             ARIMA(2, 0, 0)x(2, 1, [1], 12)   Log Likelihood                -380.565
Date:                            Sat, 22 Mar 2025   AIC                            773.131
Time:                                    11:07:02   BIC                            791.875
Sample:                                01-31-2010   HQIC                           780.738
                                     - 12-31-2024                                         
Covariance Type:                              opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.7236      0.053     13.674      0.000       0.620       0.827
ar.L2         -0.1971      0.076     -2.597      0.009      -0.346      -0.048
ar.S.L12       0.3261      0.075      4.327      0.000       0.178       0.474
ar.S.L24      -0.1699      0.067     -2.532      0.011      -0.301      -0.038
ma.S.L12      -0.9992     18.348     -0.054      0.957     -36.961      34.962
sigma2         4.5321     82.887      0.055      0.956    -157.923     166.987
===================================================================================
Ljung-Box (L1) (Q):                   0.02   Jarque-Bera (JB):               631.33
Prob(Q):                              0.87   Prob(JB):                         0.00
Heteroskedasticity (H):               0.56   Skew:                             1.49
Prob(H) (two-sided):                  0.03   Kurtosis:                        12.01
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

In [65]:
### Predictions for average snow depth per month in 2025
arima_snwd_forecast = snwd_model.predict(start='2025-01-01', end='2025-12-31')
print(arima_snwd_forecast)

2025-01    8.935727
2025-02    8.593634
2025-03    5.324302
2025-04    0.791771
2025-05   -0.049646
2025-06   -0.037059
2025-07   -0.016337
2025-08   -0.002105
2025-09    0.007045
2025-10    0.012907
2025-11    0.765462
2025-12    4.901313
Freq: M, Name: predicted_mean, dtype: float64
